# 🎓 WE4 · Notebook 04: PPO
## Landing a spacecraft on the moon

This notebook builds up to PPO the way the lecture did: fly the lander, try the
simplest policy gradient there is, watch it fail to land, and then fix it.

**Wherever you see 🎯 there is something for you to write.** Four tasks, nine
short lines in all: the loop that flies one descent, REINFORCE's objective, the
critic network, and the three lines of PPO's objective. Everything else is
already here, and every task is followed by a check that runs in about a second
and names the mistake.

By the end you will have three clips of the same lander: untrained, taught by
REINFORCE, and taught by PPO. Only the last one lands.

### Before you start

**Runtime, then Change runtime type, then choose CPU.** Not GPU.

The networks here are tiny (8 inputs, two hidden layers of 64) and the lander
takes one step at a time, so a GPU only adds launch latency to every one of half
a million tiny calls. It would be slower, and it would spend your GPU quota.

**This notebook trains for longer than most.** REINFORCE takes about four
minutes and PPO about nine. That is not padding: nine minutes is what it costs
to get a landing rather than a hover, and the difference between those two is
the whole point of the last section.

### The plan

| | |
|---|---|
| 1 | Set up. Four cells to run, nothing to read |
| 2 | Meet the environment, and 🎯 fly one whole descent yourself |
| 3 | Meet the actor, and watch an untrained one crash |
| 4 | 🎯 Write REINFORCE's objective, train for four minutes, and watch it not land |
| 5 | Why REINFORCE could not land, and 🎯 write the critic |
| 6 | 🎯 Write the three lines of PPO's objective, and check them in one second |
| 7 | Train for about nine minutes |
| 8 | Watch all three side by side |

## 1. Setup

The next four cells are plumbing: installing, importing, and defining helpers to
turn frames into video. **Run them and move on.** Nothing in them is part of the
exercise, and none of it is examinable. Click "Show code" if you are curious.

In [ ]:
#@title Run me: install
# Box2D is the physics engine LunarLander runs on. It ships as a wheel, so there
# is no compiler and no swig involved.
#
# Do NOT write gymnasium[box2d] here, tempting though it is. That extra also pulls
# pygame-ce, a fork that installs itself over Colab's preinstalled pygame and ends
# up co-owning the same import namespace, plus a swig that nothing needs because
# Box2D is already a wheel. Naming box2d directly avoids touching pygame at all.
#
# Do not add -q: it would hide a "RESTART SESSION" banner if that ever happens.
!pip install "gymnasium==1.3.0" "box2d==2.3.10"

In [ ]:
#@title Run me: version check
# Version guard. If anything goes wrong later, the output of this cell is the
# first thing to look at.
import gymnasium, numpy, torch, Box2D, pygame
print("gymnasium", gymnasium.__version__)
print("numpy    ", numpy.__version__)
print("torch    ", torch.__version__)
print("Box2D    ", Box2D.__version__)
print("pygame   ", pygame.version.ver, "(pygame-ce)" if hasattr(pygame, "IS_CE") else "(original)")
assert gymnasium.__version__.startswith("1."), "expected gymnasium 1.x"
print("\nOK")

In [ ]:
#@title Run me: imports, and one frame of the game
import numpy as np, torch, torch.nn as nn, time
import gymnasium as gym

torch.set_num_threads(1)       # measured: no slower, and it keeps runs reproducible

def make_env(render=False):
    return gym.make("LunarLander-v3",
                    render_mode="rgb_array" if render else None)

env = make_env(render=True)
print("state :", env.observation_space)
print("action:", env.action_space)

obs, _ = env.reset(seed=0)
frame = env.render()
print("one rendered frame:", frame.shape)
env.close()

import matplotlib.pyplot as plt
plt.figure(figsize=(6, 4)); plt.imshow(frame); plt.axis("off"); plt.show()

In [ ]:
#@title Run me: a helper that turns frames into video
from base64 import b64encode
from IPython.display import HTML
import imageio

def save_video(frames, path, fps=30):
    '''Write frames to mp4, falling back to gif if ffmpeg is unavailable.'''
    try:
        imageio.mimsave(path, frames, fps=fps, macro_block_size=1)
        return path
    except Exception as e:
        print("mp4 failed (%s), falling back to gif" % type(e).__name__)
        gif = path.replace(".mp4", ".gif")
        imageio.mimsave(gif, frames[::2], duration=1000 / (fps / 2), loop=0)
        return gif

def show_video(path, width=260, caption=""):
    data = b64encode(open(path, "rb").read()).decode()
    if path.endswith(".gif"):
        tag = '<img width="%d" src="data:image/gif;base64,%s">' % (width, data)
    else:
        tag = ('<video width="%d" autoplay loop controls>'
               '<source src="data:video/mp4;base64,%s" type="video/mp4"></video>') % (width, data)
    return '<figure style="margin:0 12px 0 0;text-align:center">%s<figcaption style="font:13px sans-serif;color:#555">%s</figcaption></figure>' % (tag, caption)

def play(*figs):
    display(HTML('<div style="display:flex;align-items:flex-start">%s</div>' % "".join(figs)))

## 2. The environment

Everything the lecture defined has a concrete meaning here.

| Lecture | LunarLander |
|---|---|
| **state** | 8 numbers: position x and y, velocity x and y, angle, angular velocity, and one flag per leg for ground contact |
| **action** | 4 choices: `0` do nothing, `1` fire the left engine, `2` fire the main engine, `3` fire the right engine |
| **reward** | mostly shaping: more for being closer to the pad, slower and more upright; `+10` per leg touching down; `-0.3` per frame of main engine and `-0.03` per frame of a side engine; then `+100` for landing or `-100` for crashing |
| **episode** | one descent, from the drop until the lander comes to rest, crashes, or hits the 1000 step limit |
| **return** | everything collected in one descent. **200 counts as solved** |

### How you talk to it

Two methods do everything.

`env.reset()` starts a new life and hands back the first state.

`env.step(action)` takes one action and hands back **five** things:

| what comes back | what it means | in LunarLander |
|---|---|---|
| `observation` | the new state, after your action | the 8 numbers again, updated |
| `reward` | what that single action earned | a small shaping number most frames, minus the fuel just burned, plus `+100` or `-100` on the last one |
| `terminated` | did the episode end **because of the task** | the lander came to rest, or crashed |
| `truncated` | was the episode cut short **from outside**, e.g. a time limit | yes, at 1000 steps. This one matters here |
| `info` | extra diagnostics, ignored by the algorithm | empty in this environment |

`terminated` and `truncated` are separate because they mean different things to
the algorithm. When an episode *terminates* there is genuinely no future left to
value. When it is merely *truncated* the future still exists, it just stopped
being recorded.

**Both really happen here**, and the difference is the story of this notebook. A
lander that touches down terminates after a few hundred steps. A lander that
learns to hover instead never terminates at all and is truncated at 1000, which
is exactly what a half-trained agent does. Keep an eye on the step counts.

### The five methods you need

| method | what it does | what it gives back |
|---|---|---|
| `env.reset(seed=...)` | start a new life | `(observation, info)`, a pair |
| `env.step(action)` | take one action | the five things above, in that order |
| `env.action_space.sample()` | draw a legal action at random | an action, one of `0`, `1`, `2`, `3` |
| `env.render()` | hand back the current frame as pixels | an array of shape `(400, 600, 3)` |
| `env.close()` | let the game go | nothing |

The whole of reinforcement learning is a loop over `reset` and `step`. The cell
below calls each of the first two once, so you can see exactly what comes back.

In [ ]:
env = make_env()

obs, info = env.reset(seed=0)
print("env.reset() gives back a PAIR:")
print("   observation:", np.round(obs, 3))
print("   info       :", info)

obs, reward, terminated, truncated, info = env.step(2)      # 2 = fire the main engine
print("\nenv.step(1) gives back FIVE things, in this order:")
print("   observation:", np.round(obs, 3))
print("   reward     :", reward)
print("   terminated :", terminated)
print("   truncated  :", truncated)
print("   info       :", info)

print("\nand a legal random action, for when there is no policy yet:")
print("   env.action_space.sample() ->", env.action_space.sample())
env.close()

### 🎯 Task 1: one whole descent, flown at random

The loop below is the shape of every reinforcement learning program ever written:
act, observe, add up the reward, stop when the episode ends. There are no neural
networks in it and nothing of PPO. It is only the methods in the table above.

**Fill in the four lines marked 🎯.** Everything you need is in that table.

Two details worth knowing:

- The loop is a `for` over `max_steps` rather than a `while True`, so that an
  unfinished line 4 cannot hang your notebook. The environment caps a descent at
  1000 steps of its own accord, so a correct loop never reaches this one.
- `env.reset(seed=...)` seeds the *game*, but the action space draws its random
  actions from a separate generator, so it gets its own `env.action_space.seed(...)`.
  Seeding both is what makes everyone in the room see the same life.

In [ ]:
def play_one_life(env, seed=0, max_steps=2000):
    '''Fly one whole descent choosing actions at random.

    Returns (steps, total_reward): how many steps the lander lasted, and the
    RETURN for that descent, meaning every reward it collected added up.
    '''
    obs, info = env.reset(seed=seed)
    total_reward, steps = 0.0, 0

    for _ in range(max_steps):

        # 🎯 LINE 1. Pick an action at random. There is no policy yet, so this is
        #       a four-sided die. The action space can hand you a legal one.
        action = None  # 🎯 replace this

        # 🎯 LINE 2. Take that action in the game. The five things it gives back are
        #       already unpacked for you, in the order the table above lists
        #       them, so you only need the call itself.
        obs, reward, terminated, truncated, info = None  # 🎯 replace this

        # 🎯 LINE 3. Add what this one step earned to the running total. This is the
        #       whole definition of the return: the rewards, added up.
        total_reward += None  # 🎯 replace this

        steps += 1

        # 🎯 LINE 4. Stop as soon as the descent is over, whether the lander came
        #       to rest, crashed, or was cut short from outside.
        if None:
            break

    return steps, total_reward

In [ ]:
#@title Run me: check your loop
class _Spy(gym.Wrapper):
    '''Watches every call your loop makes, so the check can be specific.'''
    def __init__(self, env):
        super().__init__(env)
        self.actions, self.rewards, self.dones = [], [], []
    def step(self, action):
        self.actions.append(action)
        out = self.env.step(action)
        self.rewards.append(float(out[1]))
        self.dones.append(bool(out[2] or out[3]))
        return out

def check_loop():
    spy = _Spy(make_env())
    spy.action_space.seed(0)
    try:
        steps, total = play_one_life(spy, seed=0, max_steps=300)
    except Exception as e:
        msg = str(e)
        if spy.actions and spy.actions[-1] is None:
            print("LINE 1 is still None, so there was no action to play.")
            print("  Ask the action space for a random legal one.")
        elif "unpack" in msg:
            print("LINE 2 is still None. Put the env.step(...) call on the right")
            print("  of the equals sign, so there are five things to unpack.")
        elif "+=" in msg or "unsupported operand" in msg:
            print("LINE 3 is still None. Add this step's reward to the total.")
        else:
            print("Your loop raised %s: %s" % (type(e).__name__, msg))
        spy.close(); return

    n = len(spy.actions)
    spy.close()

    if n == 0:
        print("FAIL: your loop never called env.step, so nothing was played."); return

    # LINE 1: were the actions random draws from the action space?
    if any(a is None for a in spy.actions):
        print("FAIL on LINE 1, the action.")
        print("  It is still None, and the environment did not complain about it.")
        print("  Ask the action space for a random legal one.")
        return
    ref = gym.spaces.Discrete(4); ref.seed(0)
    want = [int(ref.sample()) for _ in range(n)]
    taken = [int(a) for a in spy.actions]
    if taken != want:
        if len(set(taken)) == 1:
            print("FAIL on LINE 1, the action.")
            print("  Every action was %d. The lander needs to be firing at random," % taken[0])
            print("  not doing the same thing forever. Ask the action space for one.")
        else:
            print("Nearly. Your loop plays a legal random life, so the logic is right,")
            print("  but those draws came from a different generator, so your life will")
            print("  not match the room's. Use the action space's own:")
            print("  env.action_space.sample().")
        return

    # LINE 4: did it stop exactly when the life ended?
    if any(spy.dones[:-1]):
        print("FAIL on LINE 4, the stopping condition.")
        print("  You kept playing after the life had already ended.")
        print("  Both terminated and truncated mean it is over.")
        return
    if not spy.dones[-1]:
        print("FAIL on LINE 4, the stopping condition.")
        print("  Your loop ran to the %d step cap without ever breaking out," % n)
        print("  so the condition never became true. It should end the life.")
        return

    # LINE 3: is total_reward really the sum of the rewards?
    if steps != n:
        print("FAIL: steps came out as %s but env.step was called %d times." % (steps, n)); return
    want_total = sum(spy.rewards)
    if abs(total - want_total) > 1e-6:
        print("FAIL on LINE 3, the running total.")
        print("  you got %.4f, the rewards actually add up to %.4f" % (total, want_total))
        if abs(total) < 1e-9:
            print("  Nothing was ever added: the total is still its starting 0.0.")
        elif abs(total - n) < 1e-6:
            print("  You added 1 per step, which counts steps. Add what the step EARNED.")
        return

    # LINE 4 again, on an env that IS cut short from outside, so that a
    # condition testing only terminated cannot slip through.
    short = _Spy(gym.wrappers.TimeLimit(make_env(), max_episode_steps=20))
    short.action_space.seed(0)
    try:
        s2, _ = play_one_life(short, seed=0, max_steps=300)
    finally:
        short.close()
    if s2 != 20:
        print("FAIL on LINE 4, the stopping condition.")
        print("  On a life cut short from outside after 20 steps, your loop ran for")
        print("  %d. truncated ends the life exactly as terminated does, and the" % s2)
        print("  condition has to accept either one.")
        return

    print("PASS. Your loop plays a whole life and returns %d steps, return %.2f." % (steps, total))
    print("      That return is one sample of exactly the number PPO is built to raise.")

check_loop()

Now run it for real. The cap is generous here; a random agent never gets near it.

In [ ]:
env = make_env()
env.action_space.seed(0)          # so everyone's coin flips match

steps, total_reward = play_one_life(env, seed=0)
env.close()

print("a random pilot lasted %d steps (%.1f seconds)" % (steps, steps / 50))
print("its RETURN for that descent was %.2f" % total_reward)
print("\nthat return is the number the whole algorithm is trying to make bigger.")

Deeply negative, and over in a second and a bit. Firing engines at random burns
fuel in every direction at once, so the lander tumbles away from the pad and hits
the ground hard, collecting the `-100` for crashing on top of everything the
shaping already took off. Remember that `200` counts as solved. Everything from
here is about closing that gap.

## 3. The actor, and an untrained one

The **actor** is the policy. It answers *"how much do I like each action here?"*,
so a state goes in and **one number per action** comes out. That is the only
network needed for the next section; a second one arrives when we find out what
this one cannot do on its own.

One line in the next cell is worth reading slowly, because it is how a network
turns into a decision:

```python
a = int(torch.argmax(actor(torch.as_tensor(obs, dtype=torch.float32))))
```

Inside out:

1. `torch.as_tensor(obs, dtype=torch.float32)` turns the 8 numbers into a tensor
   the network can accept. The cast matters: the weights are float32 and torch
   will not quietly mix types.
2. `actor(...)` pushes them through the network. Out come **four numbers, one per
   action**. They are *logits*, unbounded scores, not probabilities. Higher means
   the policy likes that action more.
3. `torch.argmax(...)` takes the index of the largest, so `0`, `1`, `2` or `3`.
4. `int(...)` unwraps it to a plain integer, which is what `env.step` wants.

It is wrapped in `with torch.no_grad():`, which tells torch **not to record any
of this for training**. While playing we only want the answer, not the machinery
for differentiating it. That is also why the log-probabilities stored during
play are frozen constants later: they were computed with the gradient turned off.

Nothing to write here. Run it, and watch an actor that has never learned
anything.

In [ ]:
class Actor(nn.Module):
    '''Given a state, how much do I like each action?'''

    def __init__(self, n_obs=8, n_act=4):
        super().__init__()
        # n_obs numbers in, two hidden layers of 64 with tanh, and n_act numbers
        # out: one score per action.
        self.net = nn.Sequential(nn.Linear(n_obs, 64), nn.Tanh(),
                                 nn.Linear(64, 64),    nn.Tanh(),
                                 nn.Linear(64, n_act))

    def forward(self, obs):
        return self.net(obs)                    # logits, one per action

    def dist(self, obs):
        '''The policy itself: those scores, turned into a choice we can sample.'''
        return torch.distributions.Categorical(logits=self.net(obs))


def rollout(actor, seed=0, max_frames=1000):
    '''Fly one descent with the actor's best-guess action and record the frames.'''
    env = make_env(render=True)
    obs, _ = env.reset(seed=seed)
    frames, total = [], 0.0
    for _ in range(max_frames):
        frames.append(env.render())
        with torch.no_grad():
            a = int(torch.argmax(actor(torch.as_tensor(obs, dtype=torch.float32))))
        obs, r, term, trunc, _ = env.step(a)
        total += float(r)
        if term or trunc: break
    env.close()
    return frames, total, "LANDED" if total >= 200 else "did not land"

In [ ]:
SEED = 0
torch.manual_seed(SEED); np.random.seed(SEED)

untrained_actor = Actor()
frames_before, ret_before, verdict_before = rollout(untrained_actor, seed=SEED)
print("untrained: lasted %d frames (%.1f seconds), return %.2f, %s"
      % (len(frames_before), len(frames_before) / 50, ret_before, verdict_before))

before_path = save_video(frames_before, "before.mp4", fps=50)
play(show_video(before_path, 320, "untrained: return %.2f" % ret_before))

It picks the same action every frame, because an untrained network has an
arbitrary opinion and follows it without wavering. The lander drops like a stone
and breaks up on impact, about a second in.

Its return is around `-119`. That happens to be better than the random pilot's
`-190`, and not because it is any good: firing engines at random wastes fuel in
every direction, while doing one thing consistently at least does not pay for the
other three. Neither of them is landing anything.

## 4. 🎯 Task 2: REINFORCE

REINFORCE is the simplest policy gradient there is, and it fits on one line.

Play some complete lives. For every action taken, work out $G_t$, **the return
from that step to the end of that life**. Then push up the log-probability of the
actions that were followed by a good return, and push down the ones that were
not:

$$\nabla J(\theta) \;=\; \mathbb{E}\big[\,G_t \, \nabla \log \pi_\theta(a_t \mid s_t)\,\big]$$

No value function, no probability ratio, no clipping. Just: whatever preceded a
good outcome, do more of it.

### From the formula to the line

Three things change on the way from the maths to the code.

**1. You write the quantity, not its gradient.** Torch differentiates for you, so
what you type is the thing whose derivative is the formula above. Since $G_t$ is a
fixed number that does not depend on $\theta$, that thing is simply
$G_t \log \pi_\theta(a_t \mid s_t)$: each log-probability, scaled by how well the
rest of its life went.

**2. The expectation becomes an average.** `logp` and `G` are both flat tensors
with one entry per step of the batch, so multiplying them pairs every action with
its own return. Averaging those products is the $\mathbb{E}[\cdot]$.

**3. The sign flips.** You want $J$ to go **up**, and an optimiser only ever goes
**down**.

| in the formula | in the code |
|---|---|
| $\log \pi_\theta(a_t \mid s_t)$, every step | `logp` |
| $G_t$, every step | `G` |
| $\mathbb{E}[\cdot]$ | `.mean()` |
| maximise, not minimise | a leading `-` |

### Why that actually moves the policy

Take a batch of two steps. The policy gave both the same log-probability `-0.2`,
but the first was followed by `G = +3.0` and the second by `G = -3.0`.

| | `logp` | `G` | `logp * G` |
|---|---|---|---|
| the good step | -0.2 | +3.0 | **-0.6** |
| the bad step | -0.2 | -3.0 | **+0.6** |

They average to `0.0`, so the loss is `0.0`.

Now make the **good** action a little more likely, holding the other fixed. A
log-probability closer to zero means more likely, so `logp` moves from `-0.2` to
`-0.1`. Its product becomes `-0.3`, the average `0.15`, and the loss **`-0.15`**.
Lower than before, so the optimiser is happy to go there.

Make the **bad** action more likely instead, and by the same arithmetic the loss
rises to **`+0.15`**. The optimiser refuses.

That is the whole mechanism. Nothing anywhere tells the policy which action was
good. The multiplication just makes the actions with high returns the cheap ones.

One detail that makes it work: `G` reaches you **already standardised** across the
batch, which is why it has both signs above. Without that, a batch in which every
life went slightly well would push *every* action up, differing only in how hard.

Fill in the one line below.

In [ ]:
def reinforce_loss(logp, G):
    '''
    logp : log-probability the policy gave each action it actually took
    G    : the return from that step to the end of its life, standardised
           across the batch by the caller

    Returns one number, to be minimised.
    '''

    # 🎯 Push up the log-probability of the actions that were followed by a good
    #    return, and push down the ones that were not. Three things, in order:
    #
    #      1. pair each step's log-probability with its own return   logp, G
    #      2. take the expectation over the batch                    .mean()
    #      3. negate, so that minimising the loss raises the return  -
    #
    #    Both logp and G are flat tensors of the same length, one entry per step,
    #    so step 1 needs no sum and no loop: multiplying them lines the two up.
    #
    #    Hint: the shape of the answer is   -( ... ).mean()
    loss = None  # 🎯 replace this

    return loss

### Check your line before training anything

In [ ]:
#@title Run me: check your REINFORCE line
def check_reinforce():
    logp = torch.tensor([-0.30, -1.20, -0.05, -2.00, -0.70])
    G    = torch.tensor([ 1.50, -0.80,  2.00,  0.60, -1.30])

    try:
        loss = reinforce_loss(logp, G)
    except Exception as e:
        print("Your code raised %s: %s" % (type(e).__name__, e)); return

    if loss is None:
        print("FAIL: loss is still None. Write the line."); return

    if not torch.is_tensor(loss):
        print("FAIL: the loss has to be a torch tensor, but you returned a %s."
              % type(loss).__name__)
        print("      Build it out of logp and G, so that torch can differentiate it.")
        return

    if loss.dim() != 0:
        print("FAIL: the loss has to be a single number, but you returned shape %s."
              % (tuple(loss.shape),))
        print("      One number per step is not a loss. Average over the batch.")
        return

    v = float(loss)
    if abs(v - (-0.024)) < 1e-5:
        print("PASS. Go and train.")
        print("(here logp * G is [-0.45, 0.96, -0.10, -1.20, 0.91], which averages to 0.024)")
        return

    print("Not right yet: you got %.6f, expected -0.024000" % v)
    if abs(v - 0.024) < 1e-5:
        print("  Right number, wrong sign. We want the return to go UP and an")
        print("  optimiser only goes DOWN, so the loss is the negative.")
    elif abs(v - (-0.12)) < 1e-5:
        print("  That is the sum rather than the average. Use .mean().")
    elif abs(v - 0.12) < 1e-5:
        print("  That is the sum, and the wrong sign.")
    elif abs(v - 0.85) < 1e-5:
        print("  That is the log-probabilities on their own. G is not being used,")
        print("  so this would push up every action taken, good or bad.")
    else:
        print("  Expected: -(logp * G).mean()")

check_reinforce()

### Train it

**Nothing below is marked 🎯: you do not write any of it.** It arrives in three
small pieces so that none of them is a wall, and the last one runs for about
**four minutes**.

Here is the whole thing in pseudocode first:

```
until 501,760 steps of the game have been played:

  1. PLAY whole descents, until at least 2048 steps are in hand.
     WHOLE descents, because REINFORCE cannot score an action until it
     has seen how the descent it belonged to turned out.

  2. SCORE
     for each life, walk BACKWARDS through it:
        G(t) = reward(t) + discount x G(t+1)
     then standardise G across the batch

  3. IMPROVE
     ONE gradient step:
        loss = YOUR ONE LINE
             - 0.01 x (actor's entropy)      # pay it to stay undecided

  throw the batch away
```

**One** gradient step per 2048 steps of play. Remember that number: PPO's will be
128, off the same data.

#### First, the scoring

$G_t$ is the return from step $t$ to the end of that descent. Adding it up
forwards would mean a fresh sum for every step. Walking **backwards** gets all of
them in one pass, because each step's answer is just its own reward plus the
discounted answer of the step after it:

$$G_t \;=\; r_t + \gamma\, G_{t+1}$$

The discount here is $\gamma = 0.999$, not the `0.99` you may have seen in the
lecture, and on this task that single digit decides everything. At `0.99` a
reward is worth half as much 69 steps later and almost nothing 300 steps later,
so the `+100` for touching down, which is several hundred steps away, is worth
less **right now** than the `0.3` a frame the main engine costs to get there.
Under `0.99` the arithmetic says: do not land. At `0.999` the landing bonus
survives the trip and the arithmetic changes its mind.

In [ ]:
GAMMA, ENT_COEF = 0.999, 0.01                    # both shared with PPO later
RF_STEPS, RF_BATCH, RF_LR = 501_760, 2048, 1e-3

def returns_to_go(rewards, gamma=GAMMA):
    '''G for every step of one life: what the rest of that life was worth.'''
    g, out = 0.0, []
    for r in reversed(rewards):
        g = r + gamma * g
        out.append(g)
    return out[::-1]                             # back into playing order

#### Then the playing, which you have already written

The cell below is the loop from **Task 1**. Three things changed, and they are
the three lines marked `# <-- new`: the coin flip became the actor, what happened
gets written down instead of thrown away, and it keeps starting fresh lives until
the batch is full.

In [ ]:
def collect_lives(actor, env, min_steps):
    '''Fly whole descents until at least min_steps have been collected.'''
    O, A, G, ep_returns = [], [], [], []

    while len(A) < min_steps:                          # <-- new: keep going
        obs, _ = env.reset()
        ep_o, ep_a, ep_r, done = [], [], [], False

        while not done:
            ot = torch.as_tensor(obs, dtype=torch.float32)
            with torch.no_grad():
                a = int(actor.dist(ot).sample())       # <-- was a coin flip
            obs, r, term, trunc, _ = env.step(a)
            ep_o.append(ot); ep_a.append(a); ep_r.append(float(r))   # <-- new
            done = term or trunc

        ep_returns.append(sum(ep_r))
        O += ep_o; A += ep_a
        G += returns_to_go(ep_r)                       # score the descent just flown

    return (torch.stack(O), torch.tensor(A),
            torch.tensor(G, dtype=torch.float32), ep_returns)

Run it once with the untrained actor, just to see what comes back. Nothing is
learned here; this is only to make the two cells above concrete before they
disappear inside a training loop.

These are the **raw** returns, straight out of the descent. They are dominated by
the `-100` for crashing, which every one of these descents earns. The
standardising that the loss relies on happens in the training loop, one line
before the loss is computed, and that is what puts both signs in.

In [ ]:
env = make_env()
env.reset(seed=SEED)
O, A, G, ep_returns = collect_lives(untrained_actor, env, RF_BATCH)
env.close()

print("states :", tuple(O.shape), " (one row of 8 numbers per step)")
print("actions:", tuple(A.shape))
print("returns:", tuple(G.shape))
print("descents: %d of them, averaging %.2f" % (len(ep_returns), np.mean(ep_returns)))

print("\nG for the last 6 steps of the batch:", [round(float(x), 2) for x in G[-6:]])
print("the last one is just that step's own reward: the descent ended there,")
print("so there was no future left to add.")

#### And the loop that uses them

This is the pseudocode above, as code.

In [ ]:
def train_reinforce(seed=SEED, total_steps=RF_STEPS, batch=RF_BATCH, lr=RF_LR):
    torch.manual_seed(seed); np.random.seed(seed)
    env = make_env()
    env.reset(seed=seed)                          # seed the game too, so re-running repeats
    actor = Actor(env.observation_space.shape[0], env.action_space.n)
    opt = torch.optim.Adam(actor.parameters(), lr=lr)
    log, xs, curve, seen, t0 = [], [], [], 0, time.time()

    while seen < total_steps:
        O, A, G, ep_returns = collect_lives(actor, env, batch)   # 1. PLAY
        seen += len(A); log += ep_returns

        G = (G - G.mean()) / (G.std() + 1e-8)                    # 2. SCORE

        d = actor.dist(O)                                        # 3. IMPROVE, once
        loss = reinforce_loss(d.log_prob(A), G) - ENT_COEF * d.entropy().mean()
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(actor.parameters(), 0.5)
        opt.step()

        xs.append(seen); curve.append(float(np.mean(log[-20:])))
        if len(curve) % 10 == 0:
            print("steps %6d   mean return %6.2f   %4.0fs"
                  % (seen, curve[-1], time.time() - t0), flush=True)
    env.close()
    return actor, xs, curve

reinforce_actor, rf_xs, rf_curve = train_reinforce()

In [ ]:
frames_rf, ret_rf, verdict_rf = rollout(reinforce_actor, seed=SEED)
print("REINFORCE: lasted %d frames (%.1f seconds), return %.2f, %s"
      % (len(frames_rf), len(frames_rf) / 50, ret_rf, verdict_rf))
print("untrained: lasted %d frames (%.1f seconds), return %.2f, %s"
      % (len(frames_before), len(frames_before) / 50, ret_before, verdict_before))

rf_path = save_video(frames_rf, "reinforce.mp4", fps=50)
play(show_video(before_path, 320, "untrained: return %.2f" % ret_before),
     show_video(rf_path,     320, "after REINFORCE: return %.2f" % ret_rf))

## 5. What REINFORCE learned, and what it did not

Something real happened. The training curve you just watched climbs the whole
way, from about `-147` at the first printed line to somewhere around `+20`. The
lander has learned to slow its descent and stop tumbling.

Now look at the clip, and at its **frame count**. The untrained lander was over in
about 50 frames because it fell out of the sky. This one runs to **1000**, the
step limit, and is cut off still airborne. It has learned the easy half of the
job, which is not to crash, and it hovers there burning fuel until the clock
stops it.

`200` is the score for landing. REINFORCE has spent **the same 501,760 steps PPO
is about to get** and arrives around `+20`, still not down. It is not stuck in a
rut, the way it would be on a simpler task: it is improving steadily, and far too
slowly to finish the job.

The reason is the whole point of the next two sections.

REINFORCE multiplies each action's log-probability by $G_t$, the return of the
rest of that descent. That is **one blunt number shared by every action in the
descent**. A descent that ends in a crash pushes down all several hundred actions
in it, including the ones that were correcting the tilt. A descent that ends well
pushes all of them up, including the ones that wasted fuel. The direction is
right on average and wrong on very nearly every individual step, and the only
remedy REINFORCE has is to fly yet more descents.

That is ruinous here in a way it is not on a short task. A descent runs for
hundreds of steps, so one number has to take the credit for hundreds of
decisions, and the noise grows with the length of the episode.

Two changes get it out of that rut, and PPO makes both:

| | what changes | what it needs |
|---|---|---|
| 1 | stop asking *"how did the whole life go"* and start asking *"was this action better than this state deserved"* | something that knows what a state is worth: the **critic**, Task 3 |
| 2 | stop throwing the batch away after one gradient step | the **probability ratio**, Task 4 |

Asking whether an action was better than its state deserved has a name. The
**advantage** of an action, written $\widehat{A}_t$, is how much better it turned
out than the critic expected of the state it was taken in. It is positive when
the action beat the critic and negative when it fell short. You never compute it
yourself. The training cell builds it from the critic's values. In Task 4 you
only multiply by it, and it arrives as the argument `adv`.

### 🎯 Task 3: the critic

The **actor** answers *"how much do I like each action here?"*, so a state goes
in and **one number per action** comes out. You already have it:

```python
self.net = nn.Sequential(nn.Linear(n_obs, 64), nn.Tanh(),
                         nn.Linear(64, 64),    nn.Tanh(),
                         nn.Linear(64, n_act))
```

The **critic** answers *"how good is this state?"*, so a state goes in and
**exactly one number** comes out. One. Not one per action, because the critic
does not score actions at all: it scores the situation.

Write it. It is the same network right up to the last layer, and that last layer
is the entire conceptual difference between the two.

In [ ]:
class Critic(nn.Module):
    '''Given a state, how good is it?'''

    def __init__(self, n_obs=8):
        super().__init__()
        # 🎯 The same network as the Actor, except for how many numbers come out
        #    of the last layer. Note there is no n_act here at all: the critic
        #    never sees how many actions exist, because it does not score them.
        self.net = None  # 🎯 replace this

    def forward(self, obs):
        return self.net(obs).squeeze(-1)        # one value per state

In [ ]:
#@title Run me: check your critic
def check_critic():
    try:
        actor, critic = Actor(), Critic()
    except Exception as e:
        print("Building the networks raised %s: %s" % (type(e).__name__, e)); return

    if critic.net is None:
        print("FAIL: self.net is still None inside Critic. Write it, copying the Actor.")
        return

    try:
        states = torch.zeros(5, 8)           # five pretend states
        raw = critic.net(states)
        v = critic(states)
    except Exception as e:
        print("Your critic raised %s: %s" % (type(e).__name__, e))
        print("Check that the first layer accepts n_obs inputs.")
        return

    if tuple(raw.shape) != (5, 1):
        print("FAIL: given 5 states, your critic's last layer returned shape %s." % (tuple(raw.shape),))
        print("      It should return (5, 1): exactly one number per state.")
        if raw.shape[-1] == 4:
            print("      You gave it 4 outputs, one per action. The critic does not score")
            print("      actions, it scores the STATE, with one number however many")
            print("      actions there happen to be.")
        return

    lin = [m for m in critic.net.modules() if isinstance(m, nn.Linear)]
    got = [(l.in_features, l.out_features) for l in lin]
    if got != [(8, 64), (64, 64), (64, 1)]:
        print("FAIL: your critic's layers are %s." % (got,))
        print("      It has to be the Actor's network with a different last layer:")
        print("      8 -> 64 -> 64 -> 1.")
        return

    acts = [m for m in critic.net.modules() if not isinstance(m, (nn.Linear, nn.Sequential))]
    if not acts or not all(isinstance(m, nn.Tanh) for m in acts):
        print("FAIL: use nn.Tanh, the same activation the Actor uses.")
        print("      The critic is the Actor's network with one number out, so a")
        print("      different activation trains to a different lander from the room's.")
        return

    one = torch.zeros(8)
    print("PASS. Your critic turns a state into a single number.")
    print("      actor  on one state -> %s, one score per action" %
          [round(float(x), 4) for x in actor(one)])
    print("      critic on one state -> %.4f, one value, full stop" % float(critic(one)))

check_critic()

## 6. 🎯 Task 4: PPO's objective

The critic fixes *what* each action is scored against. This fixes *how many
times* each batch can be used. REINFORCE took **one** gradient step per 2048
steps of play. The three lines below buy **128** gradient steps off those same
2048, and not one extra frame of the game is played for them.

### Step 1: the probability ratio

The 2048 steps were played by the policy **as it was before this update**. As soon
as you improve the policy once, that data was collected by somebody who no longer
exists. The probability ratio is how you keep using it anyway:

$$\rho_t(\theta) \;=\; \frac{\pi_\theta(a_t \mid s_t)}{\pi_{\theta_{\mathrm{old}}}(a_t \mid s_t)}$$

It asks: **how much more, or less, often would the policy I am building now have
taken this action?**

- $\rho_t > 1$: the new policy favours this action more than the collector did.
  The sample is more representative of the new policy, so it counts for more.
- $\rho_t < 1$: the new policy has moved away from this action. The sample says
  less about the new policy, so it counts for less.
- $\rho_t = 1$: the two policies agree here, and nothing is reweighted.

You are given the logarithms, `new_logp` and `old_logp`. A ratio is the
exponential of the difference of logarithms.

### Step 2: the clipped ratio

Left alone, that ratio is unbounded, and one sample could drag the policy
anywhere. So cap it:

$$\bar{\rho}_t \;=\; \operatorname{clip}(\rho_t,\; 1-\epsilon,\; 1+\epsilon)$$

If it is too large, it becomes $1+\epsilon$. If it is too small, it becomes
$1-\epsilon$. In torch this is `torch.clamp(x, lo, hi)`.

### Step 3: keep the pessimistic one

Now there are two possible multipliers for the same sample, both carrying the
advantage $\widehat{A}_t$ from the last section: the honest one
$\rho_t \widehat{A}_t$, and the capped one $\bar{\rho}_t \widehat{A}_t$.
PPO **takes the smaller of the two**:

$$L \;=\; \operatorname{average}\Big[\min\big(\rho_t \widehat{A}_t,\;\; \bar{\rho}_t \widehat{A}_t\big)\Big]$$

Taking the **minimum** means always believing the less flattering of the two
estimates. That one word is what makes the clipping do the right thing:

| advantage positive, and the ratio has | the clipped term is | `min` keeps | effect |
|---|---|---|---|
| grown past $1+\epsilon$ | smaller | the cap | no further reward for pushing |
| fallen below $1-\epsilon$ | larger | the honest $\rho_t \widehat{A}_t$ | the gradient still flows, so it can come back |

So clipping stops a sample pushing **further away** from $\rho = 1$, but never
stops it coming **back**. With `max` instead of `min` you would get exactly the
wrong behaviour in both rows.

One last thing: we want to **maximise** $L$, but optimisers **minimise**, so
return the negative.

Fill in the three lines below.

In [ ]:
CLIP_EPS = 0.2

def ppo_losses(new_logp, old_logp, adv, clip_eps=CLIP_EPS):
    '''
    new_logp : log of the probability the CURRENT policy gives the action taken
    old_logp : log of the probability the policy that COLLECTED the data gave it
    adv      : the advantage of each step, positive where the action beat
               what the critic expected of the state it was taken in

    Returns (ratio, policy_loss).
    '''

    # 🎯 STEP 1. The probability ratio.
    #            Hint: exp(log a - log b) = a / b
    ratio = None  # 🎯 replace this

    # 🎯 STEP 2. The same ratio, capped below at 1-clip_eps and above at 1+clip_eps.
    #            Hint: torch.clamp(x, lo, hi)
    clipped_ratio = None  # 🎯 replace this

    # 🎯 STEP 3. Two candidate multipliers, ratio*adv and clipped_ratio*adv.
    #            Keep the SMALLER of the two for each step, average over the
    #            batch, and negate so it can be minimised.
    #            Hint: -torch.min(A, B).mean()
    policy_loss = None  # 🎯 replace this

    return ratio, policy_loss

### Check your three lines, before training anything

This runs in about a second on fixed numbers, and it names the specific
mistake. Do not start the nine-minute training run until it says PASS.

In [ ]:
#@title Run me: check your three lines
def check():
    new_logp = torch.tensor([-0.30, -1.20, -0.05, -2.00, -0.70])
    old_logp = torch.tensor([-0.50, -0.90, -0.05, -1.10, -1.40])
    adv      = torch.tensor([ 1.50, -0.80,  2.00,  0.60, -1.30])

    try:
        ratio, loss = ppo_losses(new_logp, old_logp, adv, 0.2)
    except Exception as e:
        if "ambiguous" in str(e):
            print("That is Python's built-in min, which cannot compare two tensors.")
            print("Use torch.min(A, B), which compares them element by element.")
            return
        print("Your code raised %s: %s" % (type(e).__name__, e)); return

    if ratio is None or loss is None:
        print("FAIL: something is still None. All three steps need replacing."); return

    want_ratio = torch.tensor([1.221403, 0.740818, 1.0, 0.40657, 2.013753])
    if not torch.allclose(ratio, want_ratio, atol=1e-4):
        print("FAIL on STEP 1, the ratio.")
        print("  you     :", [round(float(x), 4) for x in ratio])
        print("  expected:", [round(float(x), 4) for x in want_ratio])
        print("  It must be 1.0 wherever new_logp equals old_logp. Check the order")
        print("  of the subtraction: it is new minus old.")
        return

    if not torch.is_tensor(loss):
        print("FAIL: the loss has to be a torch tensor, but you returned a %s."
              % type(loss).__name__)
        print("      Build it out of ratio, clipped_ratio and adv, so that torch")
        print("      can differentiate it.")
        return

    if loss.dim() != 0:
        print("FAIL: STEP 3 returned shape %s, one number per step."
              % (tuple(loss.shape),))
        print("      A loss is a single number. Average over the batch with .mean().")
        return

    v = float(loss)
    if abs(v - (-0.157213)) < 1e-4:
        _, loss2 = ppo_losses(new_logp, old_logp, adv, 0.1)
        if abs(float(loss2) - (-0.111213)) > 1e-4:
            print("Nearly: the number is right for clip_eps=0.2, but with clip_eps=0.1")
            print("  you return the clip_eps=0.2 answer, so the bounds are typed in")
            print("  rather than built from clip_eps. Use 1 - clip_eps and 1 + clip_eps.")
            return
        print("PASS. All three steps are right. Go and train.")
        print("(for reference, the clipped ratios here are [1.2, 0.8, 1.0, 0.8, 1.2])")
        return

    print("STEP 1 is right, but the final loss is not.")
    print("  you got %.6f, expected -0.157213" % v)
    if abs(v - 0.157213) < 1e-4:
        print("  Right number, wrong sign. Negate it.")
    elif abs(v - (-0.43189)) < 1e-4:
        print("  That is max() instead of min(). PPO keeps the SMALLER of the two")
        print("  candidates, the pessimistic one.")
    elif abs(v - (-0.173103)) < 1e-4:
        print("  That is ratio*adv only: STEP 2 is not being used in STEP 3.")
    elif abs(v - (-0.416)) < 1e-3:
        print("  That is clipped_ratio*adv only: you need BOTH candidates.")
    else:
        print("  Expected: -torch.min(ratio*adv, clipped_ratio*adv).mean()")

check()

## 7. Train

**Nothing here is marked 🎯: you do not write anything.** Read the shape, then
run it. The loop below is ordinary PPO, and it calls the `ppo_losses` you just
wrote. It arrives in the same three pieces as REINFORCE, and the last one runs
for about **nine minutes**.

Two numbers are printed. The **mean return** is the one that has to climb past
where REINFORCE ran out of road. The **mean length** is the more interesting one:
watch it rise towards the 1000 step limit, sit there for most of the run, and
then fall away sharply near the end. That fall is the lander giving up hovering
and starting to land.

The first two pieces are worth comparing against their counterparts:
`advantages` against `returns_to_go`, and `collect_steps` against
`collect_lives`. Those two differences **are** the difference between the
algorithms.

It gets the same budget REINFORCE got: 501,760 steps of the game, the same
network, the same discount and the same entropy bonus. REINFORCE keeps the larger
learning rate that suits it (`1e-3` against PPO's `3e-4`); apart from that, what
differs is the objective, and what the objective lets you do with the data.

Here is the whole thing in pseudocode, three phases repeated 245 times:

```
repeat 245 times:                     # one "update"

  1. PLAY
     play 2048 steps with the CURRENT actor, and for each step write down:
        state, action, log-probability of that action, the critic's value,
        the reward, and whether the life ended

  2. SCORE
     walk BACKWARDS through those 2048 steps:
        surprise  = reward + discount x value(next state) - value(this state)
        advantage = surprise + discount x lambda x advantage(next step)

  3. IMPROVE
     repeat 4 times:                  # four passes over the SAME 2048 steps
        for each minibatch of 64:
           loss = YOUR THREE LINES                  # moves the ACTOR
                + 0.5 x (critic's error)^2          # teaches the CRITIC
                - 0.01 x (actor's entropy)          # pays the actor to stay undecided
           nudge both networks downhill

  throw the 2048 steps away: the actor has moved, so they are stale
```

That `repeat 4 times`, times 32 minibatches, is the **128** gradient steps your
three lines bought: 31,360 over the run instead of 245, off the very same 501,760
steps of the game.

#### First, the scoring

`returns_to_go` measured every action against the whole rest of its life. This
measures each step against what the critic expected instead. The **surprise** at
step $t$ is how much better the reward plus the next state's value turned out
than the value of the state you were in:

$$\delta_t \;=\; r_t + \gamma\, V(s_{t+1}) - V(s_t)$$

Then those surprises are discounted backwards exactly as the rewards were, which
is the only line this shares with `returns_to_go`.

In [ ]:
UPDATES, ROLLOUT, EPOCHS, MINIBATCH = 245, 2048, 4, 64
LAM, LR, VF_COEF = 0.98, 3e-4, 0.5          # GAMMA and ENT_COEF are REINFORCE's

def advantages(R, V, D, last_v, gamma=GAMMA, lam=LAM):
    '''How much better each action was than its state deserved.'''
    n = len(R)
    adv, run = torch.zeros(n), 0.0
    for t in reversed(range(n)):
        nextv   = last_v if t == n - 1 else V[t + 1]
        nonterm = 1.0 - D[t]                              # 0 if the life ended here
        surprise = R[t] + gamma * nextv * nonterm - V[t]
        run = surprise + gamma * lam * nonterm * run      # same backwards walk
        adv[t] = run
    return adv

#### Then the playing, with one difference that matters

`collect_lives` waited for each life to finish. This one stops after exactly 2048
steps, **even if the lander is still falling**, and that is why the two extra things
below exist:

- `state` carries the half-finished life over to the next update, so the game
  goes on from where it stopped rather than restarting.
- the critic is asked for a value at every step, because when a rollout ends
  mid-flight something has to say what the unfinished future is worth.

It also writes down `LOGP`, the log-probability the policy gave each action **at
the time it was taken**. That is the $\pi_{\theta_{old}}$ in your ratio.

In [ ]:
def collect_steps(actor, critic, env, state, n_steps):
    '''Play exactly n_steps, finished descents or not.'''
    obs, ep_ret, ep_len = state
    O = torch.zeros(n_steps, obs.shape[0]); A = torch.zeros(n_steps, dtype=torch.long)
    LOGP = torch.zeros(n_steps); R = torch.zeros(n_steps)
    D = torch.zeros(n_steps);    V = torch.zeros(n_steps)
    finished, lengths = [], []

    for t in range(n_steps):
        with torch.no_grad():
            d = actor.dist(obs); a = d.sample()
            O[t], A[t], LOGP[t], V[t] = obs, a, d.log_prob(a), critic(obs)
        nobs, r, term, trunc, _ = env.step(int(a))
        R[t], D[t] = float(r), float(term or trunc)
        ep_ret += float(r); ep_len += 1
        if term or trunc:
            finished.append(ep_ret); lengths.append(ep_len)   # how LONG, as well as how good
            ep_ret, ep_len = 0.0, 0; nobs, _ = env.reset()
        obs = torch.as_tensor(nobs, dtype=torch.float32)

    return (O, A, LOGP, R, D, V), (obs, ep_ret, ep_len), finished, lengths

#### And the loop that uses them

Same three phases as REINFORCE. The only structural difference is inside phase 3,
where the batch is walked over ten times in minibatches of 64 instead of once,
and that is what your three lines paid for.

In [ ]:
def train(seed=SEED, updates=UPDATES):
    torch.manual_seed(seed); np.random.seed(seed)     # re-seed so re-running is repeatable
    env = make_env()
    actor  = Actor(env.observation_space.shape[0], env.action_space.n)
    critic = Critic(env.observation_space.shape[0])
    params = [*actor.parameters(), *critic.parameters()]   # one optimiser, so one opt.step() moves both
    opt = torch.optim.Adam(params, lr=LR)

    obs, _ = env.reset(seed=seed)
    state = (torch.as_tensor(obs, dtype=torch.float32), 0.0, 0)
    log, loglen, curve, t0 = [], [], [], time.time()

    for update in range(updates):
        batch, state, finished, lens = collect_steps(actor, critic, env, state, ROLLOUT)
        O, A, LOGP, R, D, V = batch                             # 1. PLAY
        log += finished; loglen += lens

        with torch.no_grad(): last_v = critic(state[0])         # 2. SCORE
        adv = advantages(R, V, D, last_v)
        ret = adv + V

        idx = np.arange(ROLLOUT)                                # 3. IMPROVE, 128 times
        for _ in range(EPOCHS):
            np.random.shuffle(idx)
            for s in range(0, ROLLOUT, MINIBATCH):
                mb = idx[s:s + MINIBATCH]
                d = actor.dist(O[mb])
                a_mb = adv[mb]; a_mb = (a_mb - a_mb.mean()) / (a_mb.std() + 1e-8)

                _, policy_loss = ppo_losses(d.log_prob(A[mb]), LOGP[mb], a_mb)   # <<< your code

                value_loss = ((critic(O[mb]) - ret[mb]) ** 2).mean()
                loss = policy_loss + VF_COEF * value_loss - ENT_COEF * d.entropy().mean()
                opt.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(params, 0.5)
                opt.step()

        curve.append(float(np.mean(log[-20:])) if log else 0.0)
        if (update + 1) % 20 == 0:
            print("update %3d/%d   steps %6d   mean return %7.2f   mean length %4.0f   %4.0fs"
                  % (update + 1, updates, (update + 1) * ROLLOUT, curve[-1],
                     float(np.mean(loglen[-20:])) if loglen else 0.0, time.time() - t0), flush=True)
    env.close()
    return actor, critic, curve

trained_actor, trained_critic, curve = train()

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 3.2))
plt.plot(rf_xs, rf_curve, label="REINFORCE")
plt.plot(np.arange(1, len(curve) + 1) * ROLLOUT, curve, label="PPO")
plt.xlabel("steps of the game played"); plt.ylabel("mean return, last 20 lives")
plt.title("the same budget, two objectives")
plt.legend(); plt.grid(alpha=.3); plt.show()

## 8. Watch all three

The clips below are recorded with each actor's **best** action at every step
rather than a sampled one, so what you see is what was actually learned and not a
lucky or unlucky draw. The critic is not consulted here: it existed to judge the
actor's actions during training, not to fly.

Watch the **frame counts** as well as the returns. The untrained lander is over
in about a second. REINFORCE's runs long, because a lander that has learned not
to crash but not to land just keeps going until the 1000 step limit cuts it off.
PPO's is short again, and that is the point: it is short because the lander
arrives, touches down and stops.

In [ ]:
frames_after, ret_after, verdict_after = rollout(trained_actor, seed=SEED)
for name, f, r, v in [("untrained", frames_before, ret_before, verdict_before),
                      ("REINFORCE", frames_rf,     ret_rf,     verdict_rf),
                      ("PPO      ", frames_after,  ret_after,  verdict_after)]:
    print("%s: %4d frames (%4.1f s), return %7.2f   %s"
          % (name, len(f), len(f) / 50, r, v))

after_path = save_video(frames_after, "after.mp4", fps=50)
play(show_video(before_path, 250, "untrained: %.0f" % ret_before),
     show_video(rf_path,     250, "REINFORCE: %.0f" % ret_rf),
     show_video(after_path,  250, "PPO: %.0f" % ret_after))

## What to notice

**The reward never told it how to land.** It only ever scored the outcome: a
little for being nearer the pad, slower and more upright, a charge for every
frame of engine, and `+100` or `-100` at the end. Nobody wrote a rule about when
to fire which engine. The landing came out of trying and keeping what worked.

**Same task, same network, same number of steps.** The gap between the second
clip and the third is not more data or a bigger model. It is what each objective
does with the data it has: REINFORCE scores every action by how the whole descent
turned out and then uses the batch once; PPO scores each action against what the
critic thought the state was worth, and takes 128 gradient steps from the same
batch.

**Hovering came before landing.** Look back at the episode length in the training
log. It climbs for most of the run, up towards the 1000 step limit, and only near
the end does it fall away while the return jumps. That is the agent discovering
that not crashing is worth a lot, sitting in that discovery for a long time, and
eventually finding that touching down is worth more. The discount is what made
the second half possible: at `0.99` the landing bonus is too far away to be worth
the fuel, and the run would have stopped at hovering.

**The loop never changed.** The one you wrote in Task 1 is the same loop under
both algorithms. All that happened is that the random action became the actor,
and the numbers it threw away got written down and learned from.

**It is not finished.** Nine minutes of training on a laptop-sized machine buys a
lander that arrives, not a good pilot. Published agents train for tens of
millions of steps across many environments at once.

### After the session

- Give REINFORCE three times the budget: `train_reinforce(total_steps=1_500_000)`.
  It keeps creeping up and it still does not land, and it costs three times the
  four minute run. The problem is the credit assignment, not the patience.
- Put the discount back to the lecture's value: `GAMMA = 0.99`, re-run the cell
  that defines it, then `train()`. Measured: the lander learns to hover and stops
  there, 0 landings in 5 greedy descents, most of them truncating at the 1000
  step limit. The `+100` for touching down is several hundred steps away, and at
  `0.99` that is worth less today than the fuel it takes to get there. This is
  the sharpest lesson in the notebook and it costs one digit.
- Turn the clipping off. Scroll back to the cell that begins `CLIP_EPS = 0.2`,
  change it to `1000.0`, and run **that cell** again before running `train()`.
  Running it again is the part that matters: `clip_eps` takes its default when
  the function is defined, so a `CLIP_EPS` set anywhere else is ignored.
  Clipping is what stops one oversized update from undoing the progress of every
  update before it.
- Set `ENT_COEF = 0.0`. That removes the reward for staying undecided. The policy
  often collapses onto one action early and then stops improving.
- Change `SEED` in section 3, then Runtime > Run all. The agent is not identical
  every time.